# Fink/LSST — Cepheid Variable Stars in the Deep Drilling Fields

This notebook searches for **Cepheid variable stars** detected in the **LSST Deep Drilling Fields (DDFs)**  
using the **Fink broker** alert stream and CDS/SIMBAD crossmatch metadata.

## Strategy

1. **Cone-search** each DDF via `api/v1/conesearch` (Fink/LSST API, `r:` prefix)
2. **Deduplicate** by `diaObjectId`, keep well-sampled objects (`nDiaSources >= NP_MIN`)
3. **Select Cepheids** using the `cdsxmatch` column (populated from CDS/SIMBAD crossmatch):
   - `"Cepheid"`, `"Classical Cepheid"`, `"Type II Cepheid"`
   - Also using Fink's `f:xm_gcvs_type`, `f:xm_vsx_Type` and `f:xm_simbad_otype` for completeness
4. **Download full light curves** via `api/v1/sources` + `api/v1/fp`
5. **Lomb-Scargle period search** on each Cepheid light curve
6. **Phase-fold** light curves at the best-fit period
7. **Period–Luminosity (P–L) diagram** exploration (Leavitt law)

## Why Cepheids?

Cepheids are **classical distance indicators** (Leavitt law: log P vs M).  
They are highly periodic (days to months), high-amplitude variables, making them  
ideal test cases for the Fink alert stream and the LSST photometric pipeline:
- Predictable, repeatable light curve shape (sawtooth in r/i band)
- Known period from prior catalogues (GCVS, VSX, OGLE) — allows validation
- Multi-band data from LSST enables distance modulus estimation

## Cepheid types targeted

| CDS/SIMBAD type | Description |
|-----------------|-------------|
| `Cep` / `Cepheid` | Generic Cepheid |
| `deltaCep` / `Classical Cepheid` | Classical (Type I) Cepheid — Population I |
| `WVir` / `Type II Cepheid` | Type II Cepheid — Population II (older, metal-poor stars) |
| `bCep` | β Cephei — short-period pulsators (NOT classical Cepheids) |
| `CW` / `RVTau` | W Virginis / RV Tauri — related instability strip objects |

- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- created : 2026-06-15
- last update : 2026-06-15

| Name                          | From | Type    | Documentation                                                                                                                                                                                                                                                                                                                                        |
| ----------------------------- | ---- | ------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| clf.cats_class                | Fink | int     | CATS classifier broad class prediction with the highest probability. -1= not processed, 11=SN-like, 12=Fast (e.g. KN, ulens, Novae, ...), 13=Long (e.g. SLSN, TDE, ...), 21=Periodic (e.g. RRLyrae, EB, ...), 22=Non-periodic (e.g. AGN). See https://arxiv.org/abs/2404.08798 Available from fink_broker_version 4.0 and fink_science_version 8.26.0. |
| clf.cats_score                | Fink | float   | CATS classifier highest probability (0...1). See https://arxiv.org/abs/2404.08798 Available from fink_broker_version 4.1 and fink_science_version 8.35.0.                                                                                                                                                                                              |
| clf.earlySNIa_score           | Fink | float   | Score (0...1) for the early SN Ia classifier (binary classifier). See https://arxiv.org/abs/2404.08798 Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                         |
| clf.elephant_kstest_science   | Fink | float   | hostless indicator in the science image from the ELEPHANT pipeline. See https://arxiv.org/abs/2404.18165 Available from fink_broker_version 4.1 and fink_science_version 8.34.0.                                                                                                                                                                       |
| clf.elephant_kstest_template  | Fink | float   | hostless indicator in the template image from the ELEPHANT pipeline. See https://arxiv.org/abs/2404.18165 Available from fink_broker_version 4.1 and fink_science_version 8.34.0.                                                                                                                                                                      |
| clf.snnSnVsOthers_score       | Fink | float   | Score (0...1) for the SN classifier (binary classifier) using SuperNNova. See https://arxiv.org/abs/2404.08798 Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                 |
| fink.broker_version           | Fink | string  | fink-broker schema version used to process the alert Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                                                           |
| fink.science_version          | Fink | string  | fink-science schema version used to process the alert Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                                                          |
| lsst.schema_version           | Fink | string  | LSST schema version used to generate the alert Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                                                                 |
| misc.firstDiaSourceMjdTaiFink | Fink | string  | MJD for the first detection by Rubin. Temporary replacement for diaObject.firstDiaSourceMjdTai which is not yet populated by the project Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                       |
| pred.is_cataloged             | Fink | boolean | True if the last diaSource (alert) of the diaObject (object) has a counterpart in either SIMBAD or Gaia DR3. False otherwise. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                  |
| pred.is_first                 | Fink | boolean | True if the alert is not a Solar System object and has no history (first detection at this location). Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                          |
| pred.is_sso                   | Fink | boolean | True if the diaSource is associate to a known Solar System object. False otherwise. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                            |
| pred.main_label_classifier    | Fink | int     | Main prediction from Fink classifiers for the last received alert of this object. This is currently set to the CATS prediction only (f:clf_cats_class). Subject to change. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                     |
| pred.main_label_crossmatch    | Fink | string  | Main association from various crossmatches for the last received alert of this object. This is currently set to the SIMBAD label only (f:xm_simbad_otype). Subject to change. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                  |
| xm.gaiadr3_DR3Name            | Fink | string  | Unique source designation of closest source from Gaia catalog; if exists within 1 arcsec. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                      |
| xm.gaiadr3_Plx                | Fink | double  | Absolute stellar parallax (in milli-arcsecond) of the closest source from Gaia catalog; if exists within 1 arcsec. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                             |
| xm.gaiadr3_VarFlag            | Fink | int     | Photometric variability flag from Gaia DR3. 1 if the source is variable, 0 otherwise. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                          |
| xm.gaiadr3_e_Plx              | Fink | double  | Standard error of the stellar parallax (in milli-arcsecond) of the closest source from Gaia catalog; if exists within 1 arcsec. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                |
| xm.gcvs_type                  | Fink | string  | Object type of the closest source from GCVS catalog; if exists within 1 arcsec. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                                |
| xm.legacydr8_e_zphot          | Fink | float   | Uncertainty on zphot from Legacy Surveys DR8 South Photometric Redshifts catalog - standard deviation of the normally distributed photo-z posterior. Available from fink_broker_version 4.1 and fink_science_version 8.34.0.                                                                                                                           |
| xm.legacydr8_fqual            | Fink | int     | Photo-z reliability flag from Legacy Surveys DR8 South Photometric Redshifts catalog. =1 for sources expected to have well-constrained estimates Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                               |
| xm.legacydr8_pstar            | Fink | float   | Star likelihood based on colours from GMM star-QSO classification (Legacy Surveys DR8 South Photometric Redshifts catalog) Available from fink_broker_version 4.1 and fink_science_version 8.34.0.                                                                                                                                                     |
| xm.legacydr8_zphot            | Fink | float   | Photo-z estimate from Legacy Surveys DR8 South Photometric Redshifts catalog - mean of the normally distributed photo-z posterior Available from fink_broker_version 4.1 and fink_science_version 8.34.0.                                                                                                                                              |
| xm.mangrove_2MASS_name        | Fink | string  | 2MASS source designation of closest source from Mangrove catalog; if exists within 1 arcmin. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                   |
| xm.mangrove_HyperLEDA_name    | Fink | string  | HyperLEDA source designation of closest source from Mangrove catalog; if exists within 1 arcmin. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                               |
| xm.mangrove_ang_dist          | Fink | string  | Angular distance of closest source from Mangrove catalog; if exists within 1 arcmin. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                           |
| xm.mangrove_lum_dist          | Fink | string  | Luminosity distance of closest source from Mangrove catalog; if exists within 1 arcmin. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                        |
| xm.simbad_otype               | Fink | string  | Object type of the closest source from SIMBAD database; if exists within 1 arcsec. See https://api.lsst.fink-portal.org/api/v1/classes Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                         |
| xm.spicy_class                | Fink | string  | Class name of closest source from SPICY catalog; if exists within 1.2 arcsec. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                                  |
| xm.tns_fullname               | Fink | string  | TNS name, if it exists. Available from fink_broker_version 4.1 and fink_science_version 8.36.0.                                                                                                                                                                                                                                                        |
| xm.tns_redshift               | Fink | float   | Redshift from TNS, if it exists. Available from fink_broker_version 4.1 and fink_science_version 8.36.0.                                                                                                                                                                                                                                               |
| xm.tns_type                   | Fink | string  | TNS label, if it exists. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                                                                                       |
| xm.vsx_Type                   | Fink | string  | Object type of the closest source from VSX catalog; if exists within 1 arcsec. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                                 |
| xm.x3hsp_type                 | Fink | string  | Counterpart (cross-match) to the 3HSP catalog if exists within 1 arcminute. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                                    |
| xm.x4lac_type                 | Fink | string  | Counterpart (cross-match) to the 4LAC DR3 catalog if exists within 1 arcminute. Available from fink_broker_version 4.0 and fink_science_version 8.26.0.                                                                                                                                                                                                |

## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import time
import warnings

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

from astropy.timeseries import LombScargle
from astropy.time import Time
import astropy.units as u

from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

print(f"pandas   version : {pd.__version__}")
print(f"numpy    version : {np.__version__}")

In [ ]:
# Interactive matplotlib backend (ipympl) — falls back to inline if not installed
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Fink API ──────────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org"

# ── Search parameters ─────────────────────────────────────────────────────────
NP_MIN = 50  # Minimum nDiaSources to keep an object (Cepheids need fewer detections)
CONE_RADIUS = 3600.0  # Cone search radius in arcsec (1 deg per DDF)
N_ALERTS_MAX = 10000  # Max alerts per cone search call
NC_PLOT = 20  # Max number of light curves to plot
SNR_MIN = 5.0  # Minimum flux SNR for light curve points
BANDS = list("ugrizy")

# if time slicing is required in the API
STARTTIME = datetime.strptime("2025-09-06 00:00:00", "%Y-%m-%d %H:%M:%S")
STOPTIME = datetime.strptime("2026-05-31 00:00:00", "%Y-%m-%d %H:%M:%S")
STEPDAYS = 15

print(STARTTIME, STOPTIME, STEPDAYS)


# ── Lomb-Scargle parameters ───────────────────────────────────────────────────
PERIOD_MIN_DAYS = 0.5  # Minimum period to probe (days)
PERIOD_MAX_DAYS = 100.0  # Maximum period (classical Cepheids: 1–100 d)
LS_SAMPLES_PER_PEAK = 10  # Oversampling for LS grid

# ── Cepheid crossmatch filter ─────────────────────────────────────────────────
# CDS/SIMBAD otype values that correspond to Cepheid variable stars.
# The 'cdsxmatch' column in Fink alerts uses the SIMBAD object type string.
CEPHEID_CDSXMATCH_TYPES = [
    "Cepheid",
    "Classical Cepheid",
    "Type II Cepheid",
]

# SIMBAD otype strings (f:xm_simbad_otype column)
CEPHEID_SIMBAD_OTYPES = {
    "Cep",  # Generic Cepheid in SIMBAD ontology
    "deltaCep",  # Classical (delta Cep) Cepheid — Population I
    "WVir",  # W Virginis — Type II Cepheid
    "bCep",  # beta Cephei (NOT classical Cepheids — short-period pulsators)
    "RV*",  # RV Tauri — related long-period instability strip
    "SX*",  # SX Phoenicis (Population II delta Scuti analog)
}

# VSX / GCVS type strings (f:xm_vsx_Type and f:xm_gcvs_type)
CEPHEID_VSX_TYPES = {
    "DCEP",  # delta Cephei (classical)
    "CW",  # W Virginis (Type II)
    "CWA",  # W Vir subtype A
    "CWB",  # W Vir subtype B
    "DCEPS",  # short-period delta Cephei
    "CEP",  # generic Cepheid
    "CEP(B)",  # Cepheid with two periods
    "ACEP",  # anomalous Cepheid
    "BLBOO",  # BL Boo (unusual Cepheid-like)
    "RVA",  # RV Tauri subtype A
    "RVB",  # RV Tauri subtype B
    "BCEP",  # beta Cephei
}

# ── LSST Deep Drilling Fields (RA/Dec J2000) ──────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "CEPHEIDS_DDF_01"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

# ── Plot style ────────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    """Save current figure as PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Fink API wrappers

In [ ]:
# ── Dipole columns ────────────────────────────────────────────────────────────
DIPOLE_COLS = [
    "r:isDipole",
    "r:isNegative",
    "r:dipoleFitAttempted",
    "r:dipoleFluxDiff",
    "r:dipoleFluxDiffErr",
    "r:dipoleMeanFlux",
    "r:dipoleMeanFluxErr",
    "r:dipoleLength",
    "r:dipoleAngle",
    "r:dipoleNdata",
    "r:dipoleChi2",
]

# ── Flux and source columns ───────────────────────────────────────────────────
FLUX_COLS = [
    "r:ra",
    "r:dec",
    "r:midpointMjdTai",
    "r:band",
    "r:diaObjectId",
    "r:diaSourceId",
    "r:psfFlux",
    "r:psfFluxErr",
    "r:scienceFlux",
    "r:scienceFluxErr",
    "r:templateFlux",
    "r:templateFluxErr",
    "r:apFlux",
    "r:apFluxErr",
    "r:nDiaSources",
    "r:extendedness",
    "r:reliability",
    "r:visit",
    "r:detector",
    "r:x",
    "r:y",
]

# ── Crossmatch columns ────────────────────────────────────────────────────────
CROSSMATCH_COLS = [
    "f:xm_gaiadr3_DR3Name",
    "f:xm_gaiadr3_VarFlag",
    "f:xm_gaiadr3_Plx",
    "f:xm_gaiadr3_e_Plx",
    "f:xm_gaiadr3_PhotGMag",
    "f:xm_simbad_otype",
    "f:xm_legacydr8_pstar",
    "f:xm_legacydr8_zphot",
    "f:xm_legacydr8_fqual",
    "f:xm_tns_fullname",
    "f:xm_tns_type",
    "f:xm_vsx_Type",
    "f:xm_gcvs_type",
    "f:xm_mangrove_2MASS_name",
    "f:xm_mangrove_HyperLEDA_name",
    "f:clf_cats_class",
    "f:clf_cats_score",
    "f:clf_snnSnVsOthers_score",
    "f:is_sso",
]

# ── Full column string for the API call ───────────────────────────────────────
ALL_COLS = FLUX_COLS + DIPOLE_COLS + CROSSMATCH_COLS
COLUMNS_STR = ",".join(ALL_COLS)

# Compact payload for date-window requests (avoids 400 on large fields like COSMOS)
COLUMNS_SLIM = ",".join(FLUX_COLS + DIPOLE_COLS)

print(f"Total columns requested: {len(ALL_COLS)}")
print(f"  Flux+position : {len(FLUX_COLS)}")
print(f"  Dipole        : {len(DIPOLE_COLS)}")
print(f"  Crossmatch    : {len(CROSSMATCH_COLS)}")

In [ ]:
print(f"COLUMNS_STR = {COLUMNS_STR}")

In [ ]:
def _post_json(url: str, payload: dict, timeout: int = 120) -> list | dict:
    """POST JSON and return parsed response; include response body for HTTP errors."""
    r = requests.post(url, json=payload, timeout=timeout)
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        body = (r.text or "").strip()
        msg = f"{e}"
        if body:
            msg += f" | response: {body[:500]}"
        raise requests.HTTPError(msg, response=r) from e
    return r.json()


def fetch_conesearch(
    ra: float,
    dec: float,
    radius: float,
    n: int = N_ALERTS_MAX,
    columns: str | None = COLUMNS_STR,
) -> pd.DataFrame:
    """
    Cone search via /api/v1/conesearch — returns one row per alert (diaSource).

    IMPORTANT: column names must use the 'r:' or 'f:' prefix.
    The 'i:' prefix is NOT supported here and causes HTTP 500 errors.

    Parameters
    ----------
    ra, dec  : field centre (degrees)
    radius   : search radius (arcsec)
    n        : max alerts to return
    columns  : comma-separated column string (None = all columns)

    Returns
    -------
    pd.DataFrame — one row per alert, or empty DataFrame on failure.
    """
    payload = {
        "ra": ra,
        "dec": dec,
        "radius": radius,
        "n": n,
        "output-format": "json",
    }

    print("fetch_conesearch payload : ", payload)

    if columns:
        payload["columns"] = columns
    try:
        raw = _post_json(f"{FINK_API}/api/v1/conesearch", payload)
        if not raw:
            return pd.DataFrame()
        return pd.DataFrame(raw)
    except Exception as e:
        print(f"fetch_conesearch ERROR (ra={ra:.3f}, dec={dec:.3f}): {e}")
        return pd.DataFrame()


def fetch_sources(diaObjectId: int | str, columns: str | None = None) -> pd.DataFrame:
    """Fetch diaSources (direct detections) for one diaObjectId."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/sources", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_fp(diaObjectId: int | str, columns: str | None = None) -> pd.DataFrame:
    """Fetch forced photometry for one diaObjectId."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/fp", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_objects(diaObjectId: int | str, columns: str | None = None) -> pd.DataFrame:
    """Fetch aggregated object-level statistics via /api/v1/objects."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/objects", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


print("Fink API wrappers defined.")

In [ ]:
# def fetch_conesearch_sliced(
#    ra: float,
#    dec: float,
#    radius: float,
#    n: int,
#    start: str,
#    stop: str,
#    step_days: int = 15,
#    columns: str | None = COLUMNS_STR,
# ) -> pd.DataFrame:
def fetch_conesearch_sliced(
    ra: float,
    dec: float,
    radius: float,
    n: int,
    columns: str | None = COLUMNS_STR,
) -> pd.DataFrame:
    """Cone search fallback spatial adaptatif couvrant tout le cone."""
    dfs = []

    # 2) fallback spatial adaptatif
    tile_radius = min(600.0, max(250.0, radius / 3.5))  # arcsec
    step_arcsec = 0.95 * np.sqrt(2.0) * tile_radius
    nside = int(np.ceil((2.0 * radius) / step_arcsec)) + 1
    if nside % 2 == 0:
        nside += 1
    offs_arcsec = np.linspace(-radius, radius, nside)

    print(f"   fallback tiles: nside={nside}, tile_radius={tile_radius:.0f}, step~{step_arcsec:.0f}")

    tile_dfs = []
    for dx_as in offs_arcsec:
        for dy_as in offs_arcsec:
            # ignorer les centres trop loin du cone principal
            if np.hypot(dx_as, dy_as) > (radius + tile_radius):
                continue

            ra_i = ra + dx_as / 3600.0
            dec_i = dec + dy_as / 3600.0

            dfi = fetch_conesearch(
                ra_i,
                dec_i,
                tile_radius,
                N_ALERTS_MAX,
                columns,
            )

            if not dfi.empty:
                tile_dfs.append(dfi)
                nal = len(dfi)
                print(f"\t >>> tile with n_alerts = {nal}")
            time.sleep(0.15)

        if tile_dfs:
            dft = pd.concat(tile_dfs, ignore_index=True)
            if "r:diaSourceId" in dft.columns:
                dft = dft.drop_duplicates(subset="r:diaSourceId")
            dfs.append(dft)

    if not dfs:
        return pd.DataFrame()

    df_all = pd.concat(dfs, ignore_index=True)
    if "r:diaSourceId" in df_all.columns:
        df_all = df_all.drop_duplicates(subset="r:diaSourceId")
    return df_all

## 3. Utility functions (photometry + period analysis)

In [ ]:
AB_FLUX_ZERO_NJY = 3631e9  # AB zero-point in nJy


def flux_to_mag(
    flux_nJy: np.ndarray,
    flux_err_nJy: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray | None]:
    """Convert nJy flux (and optional uncertainty) to AB magnitudes."""
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO_NJY), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def filter_lc(
    df_lc: pd.DataFrame,
    mjd_col: str = "r:midpointMjdTai",
    flux_col: str = "r:psfFlux",
    ferr_col: str = "r:psfFluxErr",
    band_col: str = "r:band",
    snr_min: float = SNR_MIN,
) -> pd.DataFrame:
    """
    Filter a raw light-curve DataFrame:
    - drop low-SNR points
    - convert flux to AB magnitudes
    Works for both diaSources and forced photometry DataFrames.
    """
    df = df_lc.copy()
    required = [mjd_col, flux_col, ferr_col, band_col]
    if not all(c in df.columns for c in required):
        missing = [c for c in required if c not in df.columns]
        print(f"  [filter_lc] missing columns: {missing}")
        return pd.DataFrame()
    df[flux_col] = pd.to_numeric(df[flux_col], errors="coerce")
    df[ferr_col] = pd.to_numeric(df[ferr_col], errors="coerce")
    df[mjd_col] = pd.to_numeric(df[mjd_col], errors="coerce")
    snr = df[flux_col].abs() / df[ferr_col].replace(0, np.nan)
    df = df[snr >= snr_min].sort_values(mjd_col).reset_index(drop=True)
    df = df.dropna(subset=[flux_col, ferr_col, mjd_col]).reset_index(drop=True)
    mag, mag_err = flux_to_mag(df[flux_col].values, df[ferr_col].values)
    df["mag"] = mag
    df["mag_err"] = mag_err
    df = df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)
    return df


def lomb_scargle_period(
    mjd: np.ndarray,
    mag: np.ndarray,
    mag_err: np.ndarray,
    period_min: float = PERIOD_MIN_DAYS,
    period_max: float = PERIOD_MAX_DAYS,
    samples_per_peak: int = LS_SAMPLES_PER_PEAK,
) -> tuple[float, np.ndarray, np.ndarray, float]:
    """
    Run Lomb-Scargle on a magnitude time series and return the best period.

    Parameters
    ----------
    mjd, mag, mag_err : arrays of floats
    period_min, period_max : float — period search range in days
    samples_per_peak : int — LS frequency grid oversampling

    Returns
    -------
    best_period : float (days)
    periods     : array of probed periods
    power       : LS power spectrum
    fap         : false alarm probability at best period
    """
    mask = np.isfinite(mjd) & np.isfinite(mag) & np.isfinite(mag_err) & (mag_err > 0)
    t, y, dy = mjd[mask], mag[mask], mag_err[mask]
    if len(t) < 5:
        return np.nan, np.array([]), np.array([]), np.nan

    freq_min = 1.0 / period_max
    freq_max = 1.0 / period_min

    ls = LombScargle(t, y, dy)
    frequency, power = ls.autopower(
        minimum_frequency=freq_min,
        maximum_frequency=freq_max,
        samples_per_peak=samples_per_peak,
    )
    periods = 1.0 / frequency
    best_freq = frequency[np.argmax(power)]
    best_period = 1.0 / best_freq
    fap = ls.false_alarm_probability(power.max(), method="bootstrap", n_bootstraps=100)
    return best_period, periods, power, fap


def phase_fold(mjd: np.ndarray, period: float, t0: float | None = None) -> np.ndarray:
    """
    Compute the phase of each observation given a period.

    Parameters
    ----------
    mjd    : array of MJD times
    period : float — period in days
    t0     : float — reference epoch (default = min(mjd))

    Returns
    -------
    phase : array in [0, 1)
    """
    if t0 is None:
        t0 = np.nanmin(mjd)
    return ((mjd - t0) / period) % 1.0


print("Utility functions defined.")


def mjd_to_isot(mjd):
    """Convert an MJD float to an ISO date string YYYY-MM-DD."""
    try:
        return Time(mjd, format="mjd").isot[:10]
    except Exception:
        return str(mjd)


print("Utility functions defined.")

## 4. Cone-search all DDFs and collect all alerts

In [ ]:
if 0:
    all_alerts = []

    for field_name, (ra, dec) in DEEP_FIELDS.items():
        print(f"\nSearching {field_name:12s}  RA={ra:8.4f}  Dec={dec:+8.4f} ...")
        try:
            df_cone = fetch_conesearch(ra, dec, radius=CONE_RADIUS, n=N_ALERTS_MAX)
            if df_cone.empty:
                print(f"  -> no alerts returned")
                continue
            df_cone["field"] = field_name
            all_alerts.append(df_cone)
            print(f"  -> {len(df_cone):6d} alerts, {df_cone['r:diaObjectId'].nunique():5d} unique objects")
        except Exception as e:
            print(f"  ERROR: {e}")
        time.sleep(0.5)  # be polite to the API

    if all_alerts:
        df_all = pd.concat(all_alerts, ignore_index=True)
        print(f"\nTotal alerts : {len(df_all):,}")
        print(f"Unique objects: {df_all['r:diaObjectId'].nunique():,}")
        print(f"Columns ({len(df_all.columns)}): {list(df_all.columns)}")
    else:
        raise RuntimeError("No alerts retrieved from any DDF — check network and API.")

In [ ]:
# ── Toggle: set True to ignore cached parquet and re-fetch from API ───────────
FORCE_RELOAD = False

all_alerts: dict[str, pd.DataFrame] = {}


# loop on DDF
for field_name, (ra, dec) in DEEP_FIELDS.items():
    parquet_path = os.path.join(DIR_DATA, f"{field_name}_alerts.parquet")
    safe_name = field_name.replace("-", "_").replace(" ", "_")

    if os.path.exists(parquet_path) and not FORCE_RELOAD:
        df = pd.read_parquet(parquet_path)
        print(f"[{field_name:12s}] Loaded from cache: {len(df):6d} alerts")
    else:
        print(f"[{field_name:12s}] Fetching from API  (ra={ra:.4f}, dec={dec:.4f}) …", end=" ")
        t0 = time.time()
        df = fetch_conesearch(ra, dec, radius=CONE_RADIUS, n=N_ALERTS_MAX)
        elapsed = time.time() - t0

        if df.empty:
            print(f"→ NO DATA  ({elapsed:.1f}s)")
            all_alerts[field_name] = df
            t0 = time.time()
            # df = fetch_conesearch_sliced(
            #    ra, dec, radius=CONE_RADIUS, n=N_ALERTS_MAX, start=STARTTIME, stop=STOPTIME, step_days=STEPDAYS
            df = fetch_conesearch_sliced(ra, dec, radius=CONE_RADIUS, n=N_ALERTS_MAX)
            elapsed = time.time() - t0

        df["field"] = field_name

        for bool_col in ["r:isDipole", "r:isNegative", "r:dipoleFitAttempted"]:
            if bool_col in df.columns:
                df[bool_col] = (
                    df[bool_col]
                    .map(
                        lambda v: (
                            True
                            if str(v).strip().lower() in ("true", "1", "yes")
                            else (False if str(v).strip().lower() in ("false", "0", "no") else pd.NA)
                        )
                    )
                    .astype("boolean")
                )

        for col in [
            "r:nDiaSourcesr:psfFlux",
            "r:psfFluxErr",
            "r:scienceFlux",
            "r:scienceFluxErr",
            "r:templateFlux",
            "r:templateFluxErr",
            "r:apFlux",
            "r:apFluxErr",
            "r:dipoleFluxDiff",
            "r:dipoleMeanFlux",
            "r:dipoleLength",
            "r:dipoleAngle",
            "r:dipoleChi2",
            "r:midpointMjdTai",
            "r:ra",
            "r:dec",
        ]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        df.to_parquet(parquet_path, index=False)
        print(f"→ {len(df):6d} alerts  ({elapsed:.1f}s) → saved in {parquet_path}")

    all_alerts[field_name] = df

print("\nFetch complete.")

##  Correct bug below

In [ ]:
df_collect = []
for field_name, (ra, dec) in DEEP_FIELDS.items():
    df_collect.append(all_alerts[field_name])

if len(df_collect) > 0:
    df_all = pd.concat(df_collect, ignore_index=True)
    print(f"\nTotal alerts : {len(df_all):,}")
    print(f"Unique objects: {df_all['r:diaObjectId'].nunique():,}")
    print(f"Columns ({len(df_all.columns)}): {list(df_all.columns)}")
else:
    raise RuntimeError("No alerts retrieved from any DDF — check network and API.")

## 5. Deduplicate and keep well-sampled objects

In [ ]:
# Deduplicate: one row per diaObjectId, keep the most recent alert
# (last alert carries the most up-to-date nDiaSources count)
df_obj = (
    df_all.sort_values("r:midpointMjdTai")
    .drop_duplicates(subset="r:diaObjectId", keep="last")
    .reset_index(drop=True)
)

print(f"Total unique objects (all fields): {len(df_obj):,}")

# Filter on minimum number of detections
df_obj["r:nDiaSources"] = pd.to_numeric(df_obj["r:nDiaSources"], errors="coerce")
df_obj = df_obj[df_obj["r:nDiaSources"] >= NP_MIN].reset_index(drop=True)
print(f"Objects with nDiaSources >= {NP_MIN}: {len(df_obj):,}")

# Show available crossmatch columns
xm_cols = [c for c in df_obj.columns if "xm" in c or "cdsxmatch" in c.lower()]
print(f"\nCrossmatch columns: {xm_cols}")

## 6. Select Cepheids via CDS crossmatch

We use three complementary crossmatch columns:
- `cdsxmatch` (or `f:cdsxmatch`) — direct CDS crossmatch type string
- `f:xm_simbad_otype` — SIMBAD object type from Fink xmatch pipeline
- `f:xm_gcvs_type` + `f:xm_vsx_Type` — variable star catalogues

In [ ]:
# ── 6a. Filter by cdsxmatch column ───────────────────────────────────────────
# Look for the column under various possible names
cds_col = None
for candidate in ("cdsxmatch", "f:cdsxmatch", "r:cdsxmatch"):
    if candidate in df_obj.columns:
        cds_col = candidate
        break

if cds_col:
    df_cepheids_cds = df_obj[df_obj[cds_col].isin(CEPHEID_CDSXMATCH_TYPES)].copy()
    print(f"Via cdsxmatch column '{cds_col}': {len(df_cepheids_cds)} Cepheids")
    print(df_cepheids_cds[cds_col].value_counts())
else:
    df_cepheids_cds = pd.DataFrame()
    print("Column 'cdsxmatch' NOT found in conesearch results.")
    print(f"Available columns: {list(df_obj.columns)}")

In [ ]:
# ── 6b. Filter by f:xm_simbad_otype ──────────────────────────────────────────
if "f:xm_simbad_otype" in df_obj.columns:
    df_cepheids_simbad = df_obj[df_obj["f:xm_simbad_otype"].isin(CEPHEID_SIMBAD_OTYPES)].copy()
    print(f"Via f:xm_simbad_otype: {len(df_cepheids_simbad)} Cepheid-class objects")
    print(df_cepheids_simbad["f:xm_simbad_otype"].value_counts())
else:
    df_cepheids_simbad = pd.DataFrame()
    print("Column f:xm_simbad_otype not found.")

In [ ]:
# ── 6c. Filter by f:xm_vsx_Type and f:xm_gcvs_type ──────────────────────────
vsx_mask = pd.Series(False, index=df_obj.index)
gcvs_mask = pd.Series(False, index=df_obj.index)

if "f:xm_vsx_Type" in df_obj.columns:
    vsx_mask = df_obj["f:xm_vsx_Type"].isin(CEPHEID_VSX_TYPES)
    print(f"Via f:xm_vsx_Type: {vsx_mask.sum()} Cepheid-class objects")

if "f:xm_gcvs_type" in df_obj.columns:
    gcvs_mask = df_obj["f:xm_gcvs_type"].isin(CEPHEID_VSX_TYPES)
    print(f"Via f:xm_gcvs_type: {gcvs_mask.sum()} Cepheid-class objects")

df_cepheids_vsx = df_obj[vsx_mask | gcvs_mask].copy()
print(f"Combined VSX + GCVS: {len(df_cepheids_vsx)} objects")

In [ ]:
# ── 6d. Union of all Cepheid selections ──────────────────────────────────────
cep_ids = set()
for df_sel in (df_cepheids_cds, df_cepheids_simbad, df_cepheids_vsx):
    if not df_sel.empty:
        cep_ids.update(df_sel["r:diaObjectId"].tolist())

df_cepheids = df_obj[df_obj["r:diaObjectId"].isin(cep_ids)].copy().reset_index(drop=True)

print(f"\n{'=' * 60}")
print(f"Total unique Cepheids (union of all criteria): {len(df_cepheids)}")
print(f"{'=' * 60}")

if len(df_cepheids) > 0:
    display(
        df_cepheids[
            ["r:diaObjectId", "r:ra", "r:dec", "r:nDiaSources", "field"]
            + ([cds_col] if cds_col else [])
            + (["f:xm_simbad_otype"] if "f:xm_simbad_otype" in df_cepheids.columns else [])
            + (["f:xm_vsx_Type"] if "f:xm_vsx_Type" in df_cepheids.columns else [])
            + (["f:xm_gcvs_type"] if "f:xm_gcvs_type" in df_cepheids.columns else [])
        ]
    )
    # Save catalogue
    out_csv = os.path.join(DIR_DATA, "cepheids_catalogue.csv")
    df_cepheids.to_csv(out_csv, index=False)
    print(f"\nCatalogue saved: {out_csv}")
else:
    print("No Cepheids found — try expanding the search (larger radius, lower NP_MIN).")
    print("\nDistribution of SIMBAD types in the full sample:")
    if "f:xm_simbad_otype" in df_obj.columns:
        print(df_obj["f:xm_simbad_otype"].value_counts().head(30))

## 7. Overview: Cepheids per field and per type

In [ ]:
if df_cepheids.empty:
    print("No Cepheids to display.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: count per DDF field
    ax = axes[0]
    df_cepheids["field"].value_counts().sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set_xlabel("Number of Cepheids")
    ax.set_title("Cepheids per DDF field")

    # Right: count per SIMBAD type
    ax = axes[1]
    if "f:xm_simbad_otype" in df_cepheids.columns:
        df_cepheids["f:xm_simbad_otype"].value_counts().sort_values().plot.barh(ax=ax, color="darkorange")
        ax.set_xlabel("Number of objects")
        ax.set_title("Cepheid sub-types (SIMBAD otype)")
    else:
        ax.text(
            0.5, 0.5, "f:xm_simbad_otype\nnot available", ha="center", va="center", transform=ax.transAxes
        )

    plt.tight_layout()
    savefig("cepheids_overview")
    plt.show()

## 8. Download light curves for all Cepheids

In [ ]:
# Download diaSources + forced photometry for each Cepheid
lc_dict = {}  # diaObjectId -> {"src": df_src, "fp": df_fp, "meta": row}

ids_to_fetch = df_cepheids["r:diaObjectId"].tolist()
print(f"Downloading light curves for {len(ids_to_fetch)} Cepheids ...")

for i, oid in enumerate(ids_to_fetch):
    meta = df_cepheids[df_cepheids["r:diaObjectId"] == oid].iloc[0]
    try:
        df_src = fetch_sources(oid)
        df_fp = fetch_fp(oid)
        lc_dict[oid] = {"src": df_src, "fp": df_fp, "meta": meta}
        nsrc = len(df_src) if not df_src.empty else 0
        nfp = len(df_fp) if not df_fp.empty else 0
        print(f"  [{i + 1:3d}/{len(ids_to_fetch)}] {oid}  {nsrc:4d} diaSrc  {nfp:5d} fp")
    except Exception as e:
        print(f"  [{i + 1:3d}/{len(ids_to_fetch)}] {oid}  ERROR: {e}")
        lc_dict[oid] = {"src": pd.DataFrame(), "fp": pd.DataFrame(), "meta": meta}
    time.sleep(0.3)

print(f"\nDownloaded light curves for {len(lc_dict)} objects.")

## 9. Lomb-Scargle period search

In [ ]:
period_results = []  # list of dicts with period-search results

for oid, data in lc_dict.items():
    df_src = data["src"]
    meta = data["meta"]

    if df_src.empty:
        continue

    # Filter the light curve: use r-band for period search (best-sampled)
    df_filt = filter_lc(df_src)
    if df_filt.empty or len(df_filt) < 10:
        continue

    df_r = df_filt[df_filt["r:band"] == "r"]
    if len(df_r) < 8:
        # Fall back to all bands combined
        df_r = df_filt

    mjd = df_r["r:midpointMjdTai"].values
    mag = df_r["mag"].values
    mag_err = df_r["mag_err"].values

    best_period, periods, power, fap = lomb_scargle_period(mjd, mag, mag_err)

    period_results.append(
        {
            "diaObjectId": oid,
            "field": meta.get("field", ""),
            "ra": meta.get("r:ra", np.nan),
            "dec": meta.get("r:dec", np.nan),
            "nDiaSources": meta.get("r:nDiaSources", 0),
            "simbad_otype": meta.get("f:xm_simbad_otype", ""),
            "vsx_type": meta.get("f:xm_vsx_Type", ""),
            "gcvs_type": meta.get("f:xm_gcvs_type", ""),
            "best_period_d": best_period,
            "ls_fap": fap,
            "n_r_band": len(df_r),
        }
    )
    print(f"  {oid}  P = {best_period:.3f} d  FAP = {fap:.2e}  (n={len(df_r)})")

df_periods = pd.DataFrame(period_results)
print(f"\nPeriod search done for {len(df_periods)} objects.")

if not df_periods.empty:
    # Save results
    out_csv = os.path.join(DIR_DATA, "cepheids_periods.csv")
    df_periods.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")
    display(df_periods.sort_values("best_period_d"))

## 10. Phase-folded light curves

In [ ]:
def plot_phased_lc(
    oid: int,
    df_src: pd.DataFrame,
    period: float,
    meta: pd.Series,
    ax: plt.Axes | None = None,
    show_fp: bool = False,
    df_fp: pd.DataFrame | None = None,
) -> plt.Axes:
    """
    Plot a phase-folded multi-band light curve for one Cepheid.

    Parameters
    ----------
    oid    : diaObjectId
    df_src : diaSources DataFrame (raw, unfiltered)
    period : best-fit period in days
    meta   : row from df_cepheids for this object
    ax     : matplotlib Axes (created if None)
    show_fp: overlay forced photometry points if True
    df_fp  : forced photometry DataFrame (required if show_fp=True)
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4))

    df_filt = filter_lc(df_src)
    if df_filt.empty:
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center", transform=ax.transAxes)
        return ax

    t0 = df_filt["r:midpointMjdTai"].min()

    for band in BANDS:
        dfb = df_filt[df_filt["r:band"] == band]
        if dfb.empty:
            continue
        phase = phase_fold(dfb["r:midpointMjdTai"].values, period, t0)
        ax.errorbar(
            np.concatenate([phase, phase + 1]),  # show two cycles
            np.concatenate([dfb["mag"].values, dfb["mag"].values]),
            yerr=np.concatenate([dfb["mag_err"].values, dfb["mag_err"].values]),
            fmt="o",
            ms=3,
            lw=0.5,
            color=BAND_COLORS.get(band, "grey"),
            label=f"{band}",
            alpha=0.8,
        )

    # Overlay forced photometry
    if show_fp and df_fp is not None and not df_fp.empty:
        df_fp_filt = filter_lc(
            df_fp,
            mjd_col="r:midpointMjdTai",
            flux_col="r:psfFlux",
            ferr_col="r:psfFluxErr",
            band_col="r:band",
        )
        for band in BANDS:
            dfb = df_fp_filt[df_fp_filt["r:band"] == band]
            if dfb.empty:
                continue
            phase = phase_fold(dfb["r:midpointMjdTai"].values, period, t0)
            ax.errorbar(
                np.concatenate([phase, phase + 1]),
                np.concatenate([dfb["mag"].values, dfb["mag"].values]),
                fmt="s",
                ms=2,
                lw=0.5,
                color=BAND_COLORS.get(band, "grey"),
                alpha=0.4,
            )

    simbad_type = meta.get("f:xm_simbad_otype", "?")
    vsx_type = meta.get("f:xm_vsx_Type", "?")
    ax.set_xlabel("Phase")
    ax.set_ylabel("AB mag (psfFlux)")
    ax.set_xlim(0, 2)
    ax.invert_yaxis()
    ax.set_title(
        f"{oid}  |  P = {period:.3f} d\n"
        f"SIMBAD: {simbad_type}  VSX: {vsx_type}  field: {meta.get('field', '?')}",
        fontsize=8,
    )
    ax.legend(fontsize=7, ncol=3, loc="upper right")
    return ax

In [ ]:
# Plot phase-folded light curves for the NC_PLOT best-period Cepheids
if df_periods.empty:
    print("No period results available.")
else:
    # Sort by FAP (most significant periods first)
    df_plot = df_periods.dropna(subset=["best_period_d"]).sort_values("ls_fap").head(NC_PLOT)

    ncols = 3
    nrows = int(np.ceil(len(df_plot) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (_, row) in enumerate(df_plot.iterrows()):
        oid = row["diaObjectId"]
        period = row["best_period_d"]
        data = lc_dict.get(oid, {})
        if not data or data["src"].empty:
            continue
        plot_phased_lc(
            oid,
            data["src"],
            period,
            data["meta"],
            ax=axes[idx],
            show_fp=True,
            df_fp=data["fp"],
        )

    # Hide unused axes
    for ax in axes[len(df_plot) :]:
        ax.set_visible(False)

    plt.suptitle(
        f"Phase-folded light curves — Cepheids in LSST DDFs\n"
        f"(sorted by Lomb-Scargle FAP, most significant first)",
        y=1.01,
        fontsize=10,
    )
    plt.tight_layout()
    savefig("cepheids_phased_lc")
    plt.show()

## 11. Lomb-Scargle periodograms

In [ ]:
# Plot LS periodograms for the top-NC_PLOT Cepheids
if df_periods.empty:
    print("No period results available.")
else:
    df_plot = df_periods.dropna(subset=["best_period_d"]).sort_values("ls_fap").head(NC_PLOT)

    ncols = 3
    nrows = int(np.ceil(len(df_plot) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (_, row) in enumerate(df_plot.iterrows()):
        oid = row["diaObjectId"]
        period = row["best_period_d"]
        data = lc_dict.get(oid, {})
        if not data or data["src"].empty:
            continue

        df_filt = filter_lc(data["src"])
        if df_filt.empty:
            continue
        df_r = df_filt[df_filt["r:band"] == "r"]
        if len(df_r) < 8:
            df_r = df_filt

        best_period, periods, power, fap = lomb_scargle_period(
            df_r["r:midpointMjdTai"].values,
            df_r["mag"].values,
            df_r["mag_err"].values,
        )

        ax = axes[idx]
        if len(periods) > 0:
            ax.semilogx(periods, power, lw=0.8, color="steelblue")
            ax.axvline(best_period, color="red", lw=1.2, ls="--", label=f"P={best_period:.3f} d")
        ax.set_xlabel("Period (days)")
        ax.set_ylabel("LS power")
        ax.set_title(f"{oid}  FAP={fap:.1e}", fontsize=8)
        ax.legend(fontsize=7)

    for ax in axes[len(df_plot) :]:
        ax.set_visible(False)

    plt.suptitle("Lomb-Scargle periodograms — Cepheids in LSST DDFs", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("cepheids_ls_periodograms")
    plt.show()

## 12. Period distribution and Period–Luminosity diagram

In [ ]:
if df_periods.empty:
    print("No period data available for P–L diagram.")
else:
    # Get mean magnitude in each band from the object catalogue
    # We use the per-object template flux as a proxy for the mean brightness
    # (mean psfFlux over all detections, per band)
    mag_mean_r = []
    for oid in df_periods["diaObjectId"]:
        data = lc_dict.get(oid, {})
        if not data or data["src"].empty:
            mag_mean_r.append(np.nan)
            continue
        df_r = filter_lc(data["src"])
        df_r = df_r[df_r["r:band"] == "r"]
        mag_mean_r.append(np.nanmedian(df_r["mag"].values) if len(df_r) > 0 else np.nan)

    df_periods["mag_median_r"] = mag_mean_r

    df_pl = df_periods.dropna(subset=["best_period_d", "mag_median_r"]).copy()
    df_pl = df_pl[(df_pl["best_period_d"] > 0.1) & (df_pl["best_period_d"] < 200)].copy()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: period histogram
    ax = axes[0]
    ax.hist(
        df_pl["best_period_d"],
        bins=np.logspace(np.log10(0.5), np.log10(150), 30),
        color="steelblue",
        edgecolor="white",
        lw=0.5,
    )
    ax.set_xscale("log")
    ax.set_xlabel("Period (days)")
    ax.set_ylabel("Number of Cepheids")
    ax.set_title("Period distribution")

    # Right: P–L diagram (apparent, not corrected for distance/extinction)
    ax = axes[1]
    sc = ax.scatter(
        np.log10(df_pl["best_period_d"]),
        df_pl["mag_median_r"],
        c=df_pl["ls_fap"].apply(lambda x: np.clip(-np.log10(max(x, 1e-10)), 0, 10)),
        cmap="viridis",
        s=40,
        alpha=0.8,
    )
    plt.colorbar(sc, ax=ax, label="-log10(FAP)")
    ax.set_xlabel("log10(Period / days)")
    ax.set_ylabel("Median r-band AB mag (apparent)")
    ax.invert_yaxis()
    ax.set_title("Period–Luminosity diagram\n(apparent magnitudes, no distance correction)")

    plt.tight_layout()
    savefig("cepheids_PL_diagram")
    plt.show()

    # Save updated period table
    out_csv = os.path.join(DIR_DATA, "cepheids_periods_with_mag.csv")
    df_periods.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")

## 13. Sky distribution of Cepheids

In [ ]:
if df_cepheids.empty:
    print("No Cepheids to plot.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))

    # Plot DDF field centres
    for fname, (ra, dec) in DEEP_FIELDS.items():
        ax.scatter(ra, dec, marker="+", s=200, color="grey", lw=1.5, zorder=2)
        ax.text(ra + 0.5, dec + 0.3, fname, fontsize=7, color="grey")

    # Plot Cepheids, coloured by best period
    df_plot = df_cepheids.merge(
        df_periods[["diaObjectId", "best_period_d", "ls_fap"]],
        left_on="r:diaObjectId",
        right_on="diaObjectId",
        how="left",
    )
    sc = ax.scatter(
        df_plot["r:ra"].astype(float),
        df_plot["r:dec"].astype(float),
        c=np.log10(df_plot["best_period_d"].clip(0.1, 200)),
        cmap="plasma",
        s=25,
        alpha=0.8,
        zorder=3,
        label="Cepheid",
    )
    plt.colorbar(sc, ax=ax, label="log10(Period / days)")

    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    ax.set_title(f"Cepheids in LSST Deep Drilling Fields (N={len(df_cepheids)})")
    ax.legend()
    plt.tight_layout()
    savefig("cepheids_sky_map")
    plt.show()

## 14. Summary table

In [ ]:
if not df_periods.empty:
    summary_cols = [
        "diaObjectId",
        "field",
        "ra",
        "dec",
        "simbad_otype",
        "vsx_type",
        "gcvs_type",
        "best_period_d",
        "ls_fap",
        "n_r_band",
    ]
    # Add apparent magnitude if computed
    if "mag_median_r" in df_periods.columns:
        summary_cols.append("mag_median_r")

    df_summary = df_periods[summary_cols].sort_values("best_period_d").reset_index(drop=True)
    display(
        df_summary.style.format(
            {
                "best_period_d": "{:.4f}",
                "ls_fap": "{:.2e}",
                "ra": "{:.5f}",
                "dec": "{:.5f}",
                "mag_median_r": "{:.3f}",
            }
        )
    )
else:
    print("No results to summarise.")

## 15. Notes and next steps

### Diagnostics to check if no Cepheids are found

If `df_cepheids` is empty, check:
1. Is the `cdsxmatch` column present in the API response? → print `df_obj.columns`
2. What SIMBAD types are actually present? → `df_obj['f:xm_simbad_otype'].value_counts()`
3. Try a wider radius: `CONE_RADIUS = 3600.0` (1 deg)
4. Try a lower detection threshold: `NP_MIN = 5`
5. Inspect the VSX / GCVS columns directly for any pulsating variable signatures.

### Possible extensions

- **Cross-match with OGLE**: the Optical Gravitational Lensing Experiment (OGLE) provides
  Cepheid catalogues with known periods; cross-match by sky position to validate LS periods.
- **Multi-band Period–Luminosity**: run LS on each band separately and derive the W_I
  Wesenheit index (extinction-free) using `W_I = I - 1.55*(V-I)` or LSST equivalent.
- **Template fitting**: fit analytical Cepheid light curve templates (Sesar et al. 2010 or
  Jones et al. 1996) to the phase-folded data for improved period and mean-magnitude estimates.
- **Distance estimates**: apply the Leavitt law calibration (e.g. Riess et al. 2022)
  to derive individual distance moduli and compare with host-galaxy distances.
- **Type II Cepheids (W Vir)**: these are fainter and longer-period objects tracing
  Population II stellar populations — useful for probing the old stellar content of DDFs.